# Prepare a HiTS data upload: plate maps and `toload` folders

This notebook is the operator's tool for **Part C** (build the `toload` folders) and **Part E**
(clean up) of the *HiTS data upload SOP*. Run the cells top to bottom, one at a time, in Jupyter.

**Before you start**
- The Montreal share is mounted at `/Volumes/Shared/`. If it's mounted somewhere else, set the `smdc_share_root` environment variable.
- The SSH tunnel to the HiTS database is open, and `mysql_user`, `mysql80_pwd` and `mysql80_port` are set in your environment.
- This repo is installed in your Python environment (`python -m pip install -e .` from the repo root), and you opened this notebook from the `database_interactions/` folder.

This notebook only **reads** from HiTS (production `QB3_HiTS`). It writes files only into the upload folder you name below.

In [ ]:
import os
import numpy as np
import pandas as pd

import plate_maps as pm          # plate maps + toload folders
import check_grid_num as cg      # EnVision barcode / plate-order audit

pd.set_option('display.max_columns', 90)


def dr_plate_files(outdir, assay_ids):
    # Every *_DR_plate.xlsx under OUTDIR for these assay IDs, skipping loaded/
    dirs = [os.path.join(outdir, d) for d in os.listdir(outdir) if d.startswith(tuple(assay_ids))]
    for d in dirs:
        for root, _, files in os.walk(d):
            if 'loaded' in root:
                continue
            for f in files:
                if f.endswith('DR_plate.xlsx') and not f.startswith('._'):
                    yield os.path.join(root, f)

## 1. Settings (the only cell you normally edit)

- `INFILE`: the scientist's upload workbook (Info tab + plate-map tabs).
- `OUTDIR`: the upload folder on Montreal where the `toload` folders are built.
- `USERS`: people who registered the compound lots, used only as a last-resort lot guess.
- `CONTROL_DICT`: the labels the scientist used for positive and negative control wells.
- `REPLACE_DICT`: every text label that appears in the plate maps and is *not* a compound. Map it to `np.nan`.

In [ ]:
INFILE = '/Volumes/Shared/SMDC-users/<scientist>/HITS Upload/toload/<YYYYMMDD>/<workbook>.xlsx'
OUTDIR = '/Volumes/Shared/SMDC-users/<scientist>/HITS Upload/toload'

USERS = [("First", "Last")]          # whoever registered the compound lots

DOSE_RESPONSE = True                 # True for DR uploads

CONTROL_DICT = {
    "Sci_PosControl": "No Enzyme",
    "Sci_NegControl": "DMSO",
}
REPLACE_DICT = {"No Enzyme": np.nan, "DMSO": np.nan}

print(INFILE)
print(OUTDIR)

## 2. Build the plate maps and `toload` folders (SOP Part C, steps 2 to 4)

This writes `<OUTDIR>/<assay id>_<assay name>/toload_<n>/<YYYYMMDD>/`, containing `<grid_num>_DR_plate.xlsx`,
`grid_num.txt` and a copy of the raw file.

An unknown text label stops the run with a `ValueError` that names the label. Add it to `REPLACE_DICT` and run again.

**Read the log for these warnings:**
- *"Was a compound pulled from two plates?"*: run step 4 below.
- *"Getting lot info from user"*: a lot was guessed as the newest one that user registered. Check those wells by hand.

In [ ]:
info = pm.make_dr_platemaps_and_folders(
    infile=INFILE, users=USERS, basedir=OUTDIR,
    dose_response=DOSE_RESPONSE,
    control_dict=CONTROL_DICT,
    replace_dict=REPLACE_DICT,
)
info    # the workbook's Info tab, one row per destination plate

## 3. EnVision check: do the barcodes and plate order match `grid_num.txt`? (SOP Part C, step 5)

**Always run this for EnVision data.** Recent uploads loaded results onto the wrong plates because
`grid_num.txt` didn't match the plates in the instrument export.

The check compares every `grid_num.txt` under `OUTDIR` with the plates (and their barcodes) in the EnVision
CSVs next to it. It writes `folder_summary.csv` and `folder_plate_details.csv` into `OUTDIR`.

| Verdict | Meaning | What to do |
| --- | --- | --- |
| `OK` | Count, order and barcodes match | Go on |
| `ORDER_MISMATCH` | Same plates, different order | Fix the order in `grid_num.txt` (sort by name, not date) and rerun |
| `BARCODES_BLANK` | The export has no barcodes, so the order can't be confirmed | Confirm the plate order with the scientist before uploading |
| `NO_MATCH` / `REVIEW` / `NO_PLATE_BLOCKS` | Count mismatch, unreadable export or no confident match | Don't upload. Check the folder by hand |

The number and order of plates in `grid_num.txt` must match the export. Write `BLANK` for an empty plate.

In [ ]:
summary_csv = os.path.join(OUTDIR, 'folder_summary.csv')
detail_csv = os.path.join(OUTDIR, 'folder_plate_details.csv')

if os.path.exists(summary_csv):
    os.remove(summary_csv)   # never read a stale report from an earlier run

exit_code = cg.main([OUTDIR, '-o', summary_csv, '-d', detail_csv])

if not os.path.exists(summary_csv):
    raise FileNotFoundError(f'No grid_num.txt folders found under {OUTDIR}. Did step 2 run?')
summary = pd.read_csv(summary_csv)
problems = summary[summary.verdict != 'OK']
if problems.empty:
    print('All folders OK.')
else:
    print(f'{len(problems)} folder(s) need attention before upload:')
    display(problems[['folder', 'verdict', 'n_grid_entries']] if 'n_grid_entries' in problems else problems[['folder', 'verdict']])

## 4. Find compounds pulled from two source plates (only if step 2 warned)

This lists every compound that appears with more than one source plate or well in the `*_DR_plate.xlsx` files
for the assay IDs you name.

In [ ]:
ASSAY_IDS = ['<assay id>']    # e.g. ['2863', '2865']

rows = []
for path in dr_plate_files(OUTDIR, ASSAY_IDS):
    t = pd.read_excel(path, engine='openpyxl')[['SMDC_ID', 'Source plate', 'Source well']].drop_duplicates()
    t = t[t.SMDC_ID.duplicated(keep=False)]
    t['Plate'] = path
    rows.append(t)

dups = pd.concat(rows) if rows else pd.DataFrame(columns=['SMDC_ID', 'Source plate', 'Source well', 'Plate'])
dups['SPlateWell'] = dups['Source plate'] + ' ' + dups['Source well']
dups = (dups.pivot_table(index=['Plate', 'SMDC_ID'], values='SPlateWell', aggfunc=list)
            .sort_values('Plate').reset_index())
print(f'{len(dups)} duplicated compound(s)')
dups

### 4b. Fix them (edits the `*_DR_plate.xlsx` files)

For each row above, decide which source plate and well is **right**. `KEEP` gives, for each row of `dups` in order,
the position (0, 1, ...) of the right entry in its `SPlateWell` list. The other entries are removed.

Nothing is written until you set `WRITE = True`.

In [ ]:
KEEP = [0] * len(dups)    # edit: position of the RIGHT source plate/well for each row
WRITE = False

for plate in dups.Plate.unique():
    t = pd.read_excel(plate, engine='openpyxl')
    before = len(t)
    for i, row in dups[dups.Plate == plate].iterrows():
        for j, entry in enumerate(row.SPlateWell):
            if j == KEEP[i]:
                continue
            src_plate, src_well = entry.split()
            t = t[~((t.SMDC_ID == row.SMDC_ID) & (t['Source plate'] == src_plate) & (t['Source well'] == src_well))]
    print(f'{plate}: {before} -> {len(t)} rows')
    if WRITE:
        t.to_excel(plate, index=False)
print('Written.' if WRITE else 'Dry run: set WRITE = True to save.')

## 5. Fill in a control compound's lot (only if needed)

Use this when a control compound has no lot in the `*_DR_plate.xlsx` files.

In [ ]:
ASSAY_IDS = ['<assay id>']
CONTROL_LOTS = {1133707: 1}   # {SMDC_ID: lot}
WRITE = False

for path in dr_plate_files(OUTDIR, ASSAY_IDS):
    t = pd.read_excel(path, engine='openpyxl')
    for smdc, lot in CONTROL_LOTS.items():
        t.loc[t.SMDC_ID == smdc, 'Lot'] = lot
    t = t.fillna('NA')
    print(path)
    if WRITE:
        t.to_excel(path, index=False)
print('Written.' if WRITE else 'Dry run: set WRITE = True to save.')

## 6. Spot-check one source plate (SOP Part C, step 6)

In [ ]:
# First real barcode listed in the Info tab's Source_plate column ("use lots" etc. are skipped)
barcodes = [b.strip() for v in info.Source_plate.dropna().astype(str) for b in v.split(';')
            if b.strip() and not b.strip().lower().startswith(('use ', 'n.a'))]
barcode = barcodes[0]
print(barcode)
platemap = pm.make_platemaps_from_barcode(barcode=barcode, outdir=None)
display(platemap[platemap.SMDC_ID.duplicated(keep=False)])   # same compound twice on one plate?
pm.draw_plate_heatmap(df=platemap)

## Optional: tidy EnVision exports before step 3

Use these only when the loader rejects an EnVision file. **They edit files in place**, so work on a copy of `all_data`.
- Trim extra columns a user added after column 24.
- Make sure each calculated-results block is followed by 3 blank lines.
- Fill in empty Z' blocks.

In [ ]:
import raw_data_parsers as rdp

ENVISION_DIR = '<folder with the original EnVision CSVs>'
TOLOAD_DIR = '<OUTDIR>/<assay id>_<assay name>/toload_<n>'

# Trim to 25 columns (row label + 24 plate columns), written next to the originals' parent folder
dfs, filenames = rdp._envision_load_csv_files(load_dir=ENVISION_DIR)
for d, name in zip(dfs, filenames):
    d[np.arange(0, 25)].to_csv(os.path.join(os.path.dirname(ENVISION_DIR), name), index=False, header=False)

print(rdp.ensure_calc_spacing_in_dir(load_dir=TOLOAD_DIR, required_empty_lines=3))   # (seen, changed, lines added)
print(rdp.fill_zprime_blocks_in_dir(load_dir=TOLOAD_DIR))                             # (files changed, blocks)

## After the Pipeline Pilot upload: move files to `loaded/` (SOP Part E, step 2)

Pass **the same workbook** you used in step 2, because the function recomputes the folder paths from it.
Keep `delete_orig=False`. There is no dry run, and deleted originals can't be recovered.

In [ ]:
pm.clean_up_folders(infile=INFILE, basedir=OUTDIR, delete_orig=False)

## Other lookups

### Plate map from a barcode

In [ ]:
barcode = '<plate barcode>'
pm.make_platemaps_from_barcode(barcode=barcode, outdir=None)

### SMDC IDs and lots from a Biomek (plate + well) list

In [ ]:
BIOMEK_FILE = '<workbook with a Biomek tab>.xlsx'
biomek = pd.read_excel(BIOMEK_FILE, sheet_name='Biomek')
biomek = biomek[['plate', 'well']].rename(columns={'plate': 'Source plate', 'well': 'Source well'})

idlots = pm.lookup_id_and_lot_by_plate_and_well(plates=biomek['Source plate'].tolist(),
                                                 wells=biomek['Source well'].tolist())
biomek.merge(idlots, how='left')